# The Four Pillars of OOP

In [1]:
import sys
from pathlib import Path

current = Path.cwd()
for parent in [current, *current.parents]:
    if (parent / '_config.yml').exists():
        project_root = parent
        break
else:
    project_root = Path.cwd().parent.parent

sys.path.insert(0, str(project_root))

from shared import thinkpython, diagram, jupyturtle, download

sys.modules['thinkpython'] = thinkpython
sys.modules['diagram'] = diagram
sys.modules['jupyturtle'] = jupyturtle
sys.modules['download'] = download


Object-oriented programming is built on four core principles.

| Pillar | Core idea |
|--------|-----------|
| **Encapsulation** | Bundle data + behavior; hide internal state |
| **Inheritance** | Build new classes on top of existing ones |
| **Polymorphism** | Same interface, different implementations |
| **Abstraction** | Expose only what callers need, hide the rest |


## Encapsulation

**Encapsulation** in OOP basically means tow things:
- bundles data (attributes) and behavior (methods) together in one unit (class), and 
- restricts direct access to internal state (the data stored in the object). 
 
In Python, the convention is to prefix private attributes with `_` and then use `@property` to expose controlled access.

The goal: callers interact only through the *public interface*, not implementation details.

Other benefits achieved by encapsulation include:
- Abstraction: A class can hide the functional implementation and discloses only the functionalities required.
- Data integrity: Because the access restriction to the internal state, validation and control over the values assigned to variables can be achieved.

*the following example is from [geeksforgeeks](https://www.geeksforgeeks.org/python/encapsulation-in-python/).

In [ ]:
class Employee:
    def __init__(self, name, salary):
        self.name = name          # public attribute
        self.__salary = salary    # private attribute

    @property   ### This is a property decorator, which allows us to access the method as an attribute.
    def salary(self):
        return self.__salary

emp = Employee("Fedrick", 50000)
print(emp.name)
print(emp.salary)   ### this will call the salary method, but we can access it as an attribute because of the @property decorator.

Fedrick
50000


## Polymorphism

**Polymorphism** means different object types can respond to the same method call in their own way.

In the banking example below, each account type implements the same method, `month_end()`.
The caller does not need `if/elif` logic for each account type — it can just call one method name on all accounts.

In [ ]:
class CheckingAccount:
    def __init__(self, owner, balance):
        self.owner = owner
        self.balance = balance

    def month_end(self):
        self.balance -= 10   # monthly service fee


class SavingsAccount:
    def __init__(self, owner, balance, rate=0.01):
        self.owner = owner
        self.balance = balance
        self.rate = rate

    def month_end(self):
        self.balance += self.balance * self.rate   # monthly interest


class LoanAccount:
    def __init__(self, owner, balance, rate=0.015):
        self.owner = owner
        self.balance = balance
        self.rate = rate

    def month_end(self):
        self.balance += self.balance * self.rate   # interest on amount owed


accounts = [
    CheckingAccount('Alice', 1200),
    SavingsAccount('Bob', 5000),
    LoanAccount('Charlie', 2000),
]

In [ ]:
print('Before month-end:')
for account in accounts:
    print(f'{account.owner}: {account.balance:.2f}')

All objects in `accounts` provide `month_end()`, so we can apply month-end updates with one uniform loop.
No type-specific `if/elif` branches are needed.

In [ ]:
for account in accounts:
    account.month_end()   # polymorphic call

print('\nAfter month-end:')
for account in accounts:
    print(f'{account.owner}: {account.balance:.2f}')

Alice: $1200.00
Bob: $950.00
Charlie: $800.00


In this loop, `account` can be a `CheckingAccount`, `SavingsAccount`, or `LoanAccount`,
but the caller code is identical:

`account.month_end()`

Python dispatches to the correct class-specific implementation at runtime.
That is the core idea of **polymorphism**: one interface, many behaviors.

## Inheritance

The language feature most often associated with object-oriented programming is **inheritance**.
Inheritance is the ability to define a new class that is a modified version of an existing class.
In this chapter I demonstrate inheritance using classes that represent playing cards, decks of cards, and poker hands.
If you don't play poker, don't worry -- I'll tell you what you need to know.

In [5]:
import thinkpython, diagram, jupyturtle

In [ ]:
# Setup: re-create Card and Deck from the next chapter (1204) so the
# Inheritance section of this notebook is self-contained for builds.
import random

class Card:
    """Represents a standard playing card."""

    suit_names = ['Clubs', 'Diamonds', 'Hearts', 'Spades']
    rank_names = [None, 'Ace', '2', '3', '4', '5', '6', '7',
                  '8', '9', '10', 'Jack', 'Queen', 'King', 'Ace']

    def __init__(self, suit, rank):
        self.suit = suit
        self.rank = rank

    def __str__(self):
        rank_name = Card.rank_names[self.rank]
        suit_name = Card.suit_names[self.suit]
        return f'{rank_name} of {suit_name}'

    def to_tuple(self):
        return (self.suit, self.rank)

    def __eq__(self, other):
        return self.suit == other.suit and self.rank == other.rank

    def __lt__(self, other):
        return self.to_tuple() < other.to_tuple()

    def __le__(self, other):
        return self.to_tuple() <= other.to_tuple()


class Deck:

    def __init__(self, cards):
        self.cards = cards

    @staticmethod
    def make_cards():
        cards = []
        for suit in range(4):
            for rank in range(2, 15):
                card = Card(suit, rank)
                cards.append(card)
        return cards

    def __str__(self):
        res = []
        for card in self.cards:
            res.append(str(card))
        return '\n'.join(res)

    def take_card(self):
        return self.cards.pop()

    def put_card(self, card):
        self.cards.append(card)

    def shuffle(self):
        random.shuffle(self.cards)


# Initialize cards list used by later cells
cards = Deck.make_cards()


The cells below use the `%%add_method_to` cell magic (provided by the `thinkpython` module). It adds one method at a time to an existing class, so we can build up a class incrementally across multiple cells rather than writing everything in one large block. A cell that begins with `%%add_method_to Card`, for example, adds the method it contains directly to the `Card` class.

## Parents and children

Inheritance is the ability to define a new class that is a modified version of an existing class.
As an example, let's say we want a class to represent a "hand", that is, the cards held by one player.

* A hand is similar to a deck -- both are made up of a collection of cards, and both require operations like adding and removing cards.

* A hand is also different from a deck -- there are operations we want for hands that don't make sense for a deck. For example, in poker we might compare two hands to see which one wins. In bridge, we might compute a score for a hand in order to make a bid.

This relationship between classes -- where one is a specialized version of another -- lends itself to inheritance. 

To define a new class that is based on an existing class, we put the name of the existing class in parentheses.

In [55]:
class Hand(Deck):
    """Represents a hand of playing cards."""

This definition indicates that `Hand` inherits from `Deck`, which means that `Hand` objects can access methods defined in `Deck`, like `take_card` and `put_card`.

`Hand` also inherits `__init__` from `Deck`, but if we define `__init__` in the `Hand` class, it overrides the one in the `Deck` class.

In [56]:
%%add_method_to Hand

    def __init__(self, label=''):
        self.label = label
        self.cards = []

This version of `__init__` takes an optional string as a parameter, and always starts with an empty list of cards.
When we create a `Hand`, Python invokes this method, not the one in `Deck` -- which we can confirm by checking that the result has a `label` attribute.

In [57]:
hand = Hand('player 1')
hand.label

'player 1'

To deal a card, we can use `take_card` to remove a card from a `Deck`, and `put_card` to add the card to a `Hand`.

In [58]:
deck = Deck(cards)
card = deck.take_card()
hand.put_card(card)
print(hand)

Ace of Spades


Let's encapsulate this code in a `Deck` method called `move_cards`.

In [59]:
%%add_method_to Deck

    def move_cards(self, other, num):
        for i in range(num):
            card = self.take_card()
            other.put_card(card)

This method is polymorphic -- that is, it works with more than one type: `self` and `other` can be either a `Hand` or a `Deck`.
So we can use this method to deal a card from `Deck` to a `Hand`, from one `Hand` to another, or from a `Hand` back to a `Deck`.

When a new class inherits from an existing one, the existing one is called the **parent** and the new class is called the **child**. In general:

* Instances of the child class should have all of the attributes of the parent class, but they can have additional attributes.

* The child class should have all of the methods of the parent class, but it can have additional methods.

* If a child class overrides a method from the parent class, the new method should take the same parameters and return a compatible result.

This set of rules is called the "Liskov substitution principle" after computer scientist Barbara Liskov.

If you follow these rules, any function or method designed to work with an instance of a parent class, like a `Deck`, will also work with instances of a child class, like `Hand`.
If you violate these rules, your code will collapse like a house of cards (sorry).

## Specialization

Let's make a class called `BridgeHand` that represents a hand in bridge -- a widely played card game.
We'll inherit from `Hand` and add a new method called `high_card_point_count` that evaluates a hand using a "high card point" method, which adds up points for the high cards in the hand.

Here's a class definition that contains as a class variable a dictionary that maps from card names to their point values.

In [60]:
class BridgeHand(Hand):
    """Represents a bridge hand."""

    hcp_dict = {
        'Ace': 4,
        'King': 3,
        'Queen': 2,
        'Jack': 1,
    }

Given the rank of a card, like `12`, we can use `Card.rank_names` to get the string representation of the rank, and then use `hcp_dict` to get its score.

In [61]:
rank = 12
rank_name = Card.rank_names[rank]
score = BridgeHand.hcp_dict.get(rank_name, 0)
rank_name, score

('Queen', 2)

The following method loops through the cards in a `BridgeHand` and adds up their scores.

In [62]:
%%add_method_to BridgeHand

    def high_card_point_count(self):
        count = 0
        for card in self.cards:
            rank_name = Card.rank_names[card.rank]
            count += BridgeHand.hcp_dict.get(rank_name, 0)
        return count

In [63]:
# This cell makes a fresh Deck and 
# initializes the random number generator

cards = Deck.make_cards()
deck = Deck(cards)
random.seed(3)

To test it, we'll deal a hand with five cards -- a bridge hand usually has thirteen, but it's  easier to test code with small examples.

In [64]:
hand = BridgeHand('player 2')

deck.shuffle()
deck.move_cards(hand, 5)
print(hand)

4 of Diamonds
King of Hearts
10 of Hearts
10 of Clubs
Queen of Diamonds


And here is the total score for the King and Queen.

In [65]:
hand.high_card_point_count()

5

`BridgeHand` inherits the variables and methods of `Hand` and adds a class variable and a method that are specific to bridge.
This way of using inheritance is called **specialization** because it defines a new class that is specialized for a particular use, like playing bridge.

The `shuffle` method for the `BridgeHand` object is the one in `Deck`.

In [ ]:
### Exercise: Circle
#   1. Define a `Circle` class with `__init__(self, center, radius)` where `center` is a `Point`.
#   2. Add a `draw(self)` method that traces the perimeter using `jumpto` and `moveto`.
#      Hint: compute 36 evenly-spaced points with `cos` and `sin` from the `math` module.
#   3. Test polymorphism: place a `Circle` in a list alongside `line1` and `line2`,
#      call `draw` on each element using a `for` loop inside `make_turtle()`.
### Your code starts here.




### Your code ends here.

In [ ]:
### Solution
from math import cos, sin, pi

class Circle:
    def __init__(self, center, radius):
        self.center = center
        self.radius = radius

    def draw(self):
        n = 36
        jumpto(self.center.x + self.radius, self.center.y)
        for i in range(1, n + 1):
            angle = 2 * pi * i / n
            x = self.center.x + self.radius * cos(angle)
            y = self.center.y + self.radius * sin(angle)
            moveto(x, y)

center = Point(150, 75)
c = Circle(center, 50)

make_turtle()
for shape in [line1, line2, c]:
    shape.draw()

## `super()` and `issubclass()`

### `super()`

`super()` returns a proxy object that lets you call a method from a parent
class without naming it explicitly. This is especially useful in `__init__`
to ensure the parent class is properly initialized before you add child-specific
attributes.


In [ ]:
class Animal:
    def __init__(self, name, sound):
        self.name = name
        self.sound = sound

    def speak(self):
        return f'{self.name} says {self.sound}!'

class Dog(Animal):
    def __init__(self, name, breed):
        super().__init__(name, sound='Woof')  # call parent __init__
        self.breed = breed

    def info(self):
        return f'{self.name} is a {self.breed}'

d = Dog('Rex', 'Labrador')
print(d.speak())   # Rex says Woof!
print(d.info())    # Rex is a Labrador


### `issubclass()`

`issubclass(Child, Parent)` returns `True` if `Child` is a subclass of `Parent`
(or is `Parent` itself). Use it when you want to check class relationships at
runtime — analogous to `isinstance()`, but for classes rather than instances.

| Function | Checks |
|---|---|
| `isinstance(obj, cls)` | Is `obj` an instance of `cls` (or subclass)? |
| `issubclass(A, B)` | Is class `A` a subclass of `B`? |


In [ ]:
print(issubclass(Dog, Animal))   # True
print(issubclass(Animal, Dog))   # False
print(issubclass(Dog, Dog))      # True (a class is a subclass of itself)

# Common use: guarding against wrong arguments
def make_animal(cls, *args, **kwargs):
    if not issubclass(cls, Animal):
        raise TypeError(f'{cls} is not an Animal subclass')
    return cls(*args, **kwargs)

rex = make_animal(Dog, 'Rex', 'Labrador')
print(rex.speak())   # Rex says Woof!


## Abstraction

**Abstraction** means defining a clean interface and hiding implementation details.
Python's `abc` module lets you create **Abstract Base Classes** that guarantee every
subclass implements a required set of methods — without specifying *how*.

In [ ]:
from abc import ABC, abstractmethod
import math

class Shape(ABC):
    """Abstract base: every Shape must implement area() and perimeter()."""

    @abstractmethod
    def area(self): ...

    @abstractmethod
    def perimeter(self): ...

    def describe(self):
        return f'{type(self).__name__}: area={self.area():.2f}, perimeter={self.perimeter():.2f}'


class Circle(Shape):
    def __init__(self, radius): self.radius = radius
    def area(self): return math.pi * self.radius ** 2
    def perimeter(self): return 2 * math.pi * self.radius


class Rectangle(Shape):
    def __init__(self, w, h): self.w, self.h = w, h
    def area(self): return self.w * self.h
    def perimeter(self): return 2 * (self.w + self.h)


for s in [Circle(5), Rectangle(4, 6)]:
    print(s.describe())
# Shape()  →  TypeError: Can't instantiate abstract class


## Summary

The four pillars work together:

- **Encapsulation** — protect state; expose behavior through a clean interface
- **Inheritance** — reuse and extend existing classes with `class Child(Parent):`
- **Polymorphism** — write code that works on any object with the right interface
- **Abstraction** — define contracts with ABCs; callers depend on interfaces, not details
